# Anomaly Detection Pipeline v5 — Literature Features + No Stacking

**Key changes from v4 (F1=0.41 on Codabench):**

1. **Removed stacking** — stacking meta-learner overfit on 200 positives, causing precision collapse (0.68→0.32)
2. **Added literature shilling detection features** (RDMA, WDMA, DegSim, LengthVar, MeanVar) — proven highest information gain in the attack detection literature
3. **Trimmed redundant distance features** — kept only KL + chi-squared (removed JS, TV, L2 which are correlated)
4. **Stronger regularization** — reduced tree depth, increased min_child_samples
5. **Median per-fold threshold** — more robust than global optimal

**Literature features added:**
- **RDMA** (Rating Deviation from Mean Agreement): Deviation weighted by inverse item frequency
- **WDMA** (Weighted DMA): Squared inverse frequency weighting — emphasizes deviations on rare items
- **DegSim** (Degree of Similarity): Avg cosine similarity with top-20 nearest neighbors
- **LengthVar**: Profile length deviation from average
- **MeanVar**: Standardized deviation of user's mean rating from global mean


In [ ]:
import numpy as np
import pandas as pd
import warnings, json
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.ensemble import IsolationForest, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# ============================================================
# 1. LOAD DATA
# ============================================================
d1 = np.load("training_batch_with_labels.npz")
d2 = np.load("first_batch_with_labels.npz")
test_data = np.load("second_batch.npz")

train_ratings = pd.DataFrame(np.concatenate([d1["X"], d2["X"]]), columns=["user","item","rating"])
labels = pd.DataFrame(np.concatenate([d1["y"], d2["y"]]), columns=["user","label"]).set_index("user")
test_ratings = pd.DataFrame(test_data["X"], columns=["user","item","rating"])
all_ratings = pd.concat([train_ratings, test_ratings], ignore_index=True)
n_items = 1000

normal_users = labels[labels.label == 0].index
normal_ratings = train_ratings[train_ratings.user.isin(normal_users)]
normal_dist_raw = normal_ratings["rating"].value_counts(normalize=True).sort_index()
normal_probs = np.array([normal_dist_raw.get(rv, 0.001) for rv in range(6)])

g_item_stats = all_ratings.groupby("item")["rating"].agg(["mean","std","count"])
g_item_stats.columns = ["item_mean","item_std","item_count"]
g_item_stats["item_std"] = g_item_stats["item_std"].fillna(0)
g_item_pop = all_ratings.groupby("item")["rating"].count()
g_mean = all_ratings["rating"].mean()

# Average profile length for LengthVar
avg_profile_len = train_ratings.groupby("user").size().mean()

print(f"Train: {train_ratings['user'].nunique()} users ({labels['label'].sum()} anomalous)")
print(f"Test:  {test_ratings['user'].nunique()} users")


In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
def compute_features(df, all_df, svd_model=None, fit_svd=False):
    feats = []

    # ===== A. Basic Rating Statistics (same as v2) =====
    basic = df.groupby("user")["rating"].agg(
        mean_r="mean", std_r="std", min_r="min", max_r="max",
        med_r="median", n_ratings="count", sum_r="sum"
    ).fillna(0)
    basic["range_r"] = basic["max_r"] - basic["min_r"]
    feats.append(basic)

    # ===== B. Distribution Shape (v2 + selective distance) =====
    def dist_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        d["skew"] = stats.skew(r) if n >= 3 else 0
        d["kurt"] = stats.kurtosis(r) if n >= 4 else 0
        vc = pd.Series(r).value_counts(normalize=True)
        d["entropy"] = stats.entropy(vc)
        d["mode_prop"] = vc.iloc[0]
        d["mad"] = np.median(np.abs(r - np.median(r)))
        d["iqr"] = np.percentile(r, 75) - np.percentile(r, 25) if n >= 4 else 0
        d["cv"] = (np.std(r) / np.mean(r)) if np.mean(r) > 0 else 0
        for rv in range(6): d[f"pr{rv}"] = np.mean(r == rv)
        d["p_extreme"] = np.mean((r == 0) | (r == 5))
        d["p_low"] = np.mean(r <= 1)
        d["p_high"] = np.mean(r >= 4)
        d["n_distinct_r"] = len(np.unique(r))
        d["change_rate"] = np.sum(np.diff(r) != 0) / (n - 1) if n > 1 else 0
        d["concentration"] = np.sum(vc.values ** 2)

        # Distance: keep only KL + chi2 (highest gain, least redundant)
        ups = np.array([np.mean(r == rv) for rv in range(6)])
        ups_safe = np.clip(ups, 0.001, None); ups_safe = ups_safe / ups_safe.sum()
        d["kl_from_normal"] = stats.entropy(ups_safe, normal_probs)
        d["chi2_from_normal"] = np.sum((ups_safe - normal_probs)**2 / normal_probs)

        # Per-rating deviations (signed + absolute)
        d["pr0_dev"] = ups[0] - normal_probs[0]
        d["pr5_dev"] = ups[5] - normal_probs[5]
        d["pr0_abs_dev"] = abs(ups[0] - normal_probs[0])
        d["pr5_abs_dev"] = abs(ups[5] - normal_probs[5])
        d["mean_dev_abs"] = abs(np.mean(r) - 3.53)
        d["std_dev_abs"] = abs(np.std(r) - 0.93)

        return pd.Series(d)
    feats.append(df.groupby("user").apply(dist_f))

    # ===== C. Interaction Structure (same as v2) =====
    inter = df.groupby("user").agg(n_unique_items=("item", "nunique"))
    inter["repeat_ratio"] = 1 - inter["n_unique_items"] / df.groupby("user").size()
    inter["density"] = df.groupby("user").size() / n_items
    feats.append(inter)

    # ===== D. Item-Normalized Deviation (same as v2) =====
    m = df.merge(g_item_stats, left_on="item", right_index=True, how="left")
    m["item_mean"] = m["item_mean"].fillna(g_mean)
    m["item_std"] = m["item_std"].fillna(1)
    m["resid"] = m["rating"] - m["item_mean"]
    m["z_resid"] = np.where(m["item_std"] > 0, m["resid"] / m["item_std"], 0)
    m["abs_resid"] = np.abs(m["resid"])

    r_agg = m.groupby("user").agg(
        mean_resid=("resid","mean"), std_resid=("resid","std"),
        abs_mean_resid=("abs_resid","mean"), max_abs_resid=("abs_resid","max"),
        med_resid=("resid","median"),
        mean_z=("z_resid","mean"), std_z=("z_resid","std"),
        abs_mean_z=("z_resid", lambda x: np.abs(x).mean()),
        resid_q90=("abs_resid", lambda x: np.percentile(x, 90)),
        n_large_dev=("abs_resid", lambda x: (x > 2).sum()),
    ).fillna(0)
    r_agg["large_dev_ratio"] = r_agg["n_large_dev"] / df.groupby("user").size()
    feats.append(r_agg)

    # ===== E. Popularity-Aware (same as v2) =====
    mp = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp["ipop"] = mp["ipop"].fillna(1)
    mp["iw"] = 1.0 / (mp["ipop"] + 1)
    mp["wr"] = mp["rating"] * mp["iw"]
    pf = mp.groupby("user").agg(
        wr_sum=("wr","sum"), iw_sum=("iw","sum"),
        mean_ipop=("ipop","mean"), std_ipop=("ipop","std"),
        mean_lp=("ipop", lambda x: np.log1p(x).mean()),
    )
    pf["w_mean_r"] = pf["wr_sum"] / pf["iw_sum"]
    pf = pf.drop(columns=["wr_sum","iw_sum"]).fillna(0)
    feats.append(pf)

    # ===== F. Sequence Patterns (same as v2) =====
    def seq_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        if n >= 3:
            mid = n // 2
            d["drift"] = np.mean(r[mid:]) - np.mean(r[:mid])
            d["trend"] = np.polyfit(range(n), r, 1)[0]
            d["autocorr"] = np.corrcoef(r[:-1], r[1:])[0, 1] if np.std(r) > 0 else 0
            if np.isnan(d["autocorr"]): d["autocorr"] = 0
        else:
            d["drift"] = 0; d["trend"] = 0; d["autocorr"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(seq_f))

    # ===== G. Item Diversity (same as v2) =====
    def div_f(g):
        vc = pd.Series(g["item"].values).value_counts(normalize=True)
        return pd.Series({
            "item_entropy": stats.entropy(vc),
            "gini_simpson": 1 - np.sum(vc ** 2),
            "unique_ratio": len(np.unique(g["item"].values)) / n_items,
        })
    feats.append(df.groupby("user").apply(div_f))

    # ===== H. SVD (same as v2) =====
    users = sorted(df["user"].unique())
    uid_map = {u: i for i, u in enumerate(users)}
    rows = df["user"].map(uid_map).values
    cols = df["item"].values
    vals = df["rating"].values.astype(float)
    um = csr_matrix((vals, (rows, cols)), shape=(len(users), n_items))
    nc = 30
    if fit_svd or svd_model is None:
        svd_model = TruncatedSVD(n_components=nc, random_state=42)
        emb = svd_model.fit_transform(um)
    else:
        emb = svd_model.transform(um)
    svd_df = pd.DataFrame(emb, index=users, columns=[f"svd_{i}" for i in range(nc)])
    svd_df.index.name = "user"
    recon = svd_model.inverse_transform(emb)
    svd_df["svd_err"] = np.mean((um.toarray() - recon) ** 2, axis=1)
    svd_df["svd_norm"] = np.linalg.norm(emb, axis=1)
    feats.append(svd_df)

    # ===== I. Cosine Sim (same as v2) =====
    bm = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(users), n_items))
    avg = bm.mean(axis=0).A1
    sims = []
    for i in range(len(users)):
        uv = bm[i].toarray().flatten()
        d = np.dot(uv, avg); nu = np.linalg.norm(uv); na = np.linalg.norm(avg)
        sims.append(d / (nu * na) if nu > 0 and na > 0 else 0)
    feats.append(pd.DataFrame({"cos_sim_avg": sims}, index=users).rename_axis("user"))

    # ===== J. NEW: Literature Shilling Detection Features =====
    # RDMA: Rating Deviation from Mean Agreement
    # RDMA_u = (1/n_u) * sum_i(|r_ui - r̄_i| / NR_i)
    # where NR_i = number of ratings for item i (inverse frequency weighting)
    merged = df.merge(g_item_stats[["item_mean","item_count"]], left_on="item", right_index=True, how="left")
    merged["item_mean"] = merged["item_mean"].fillna(g_mean)
    merged["item_count"] = merged["item_count"].fillna(1)
    merged["rdma_contrib"] = np.abs(merged["rating"] - merged["item_mean"]) / merged["item_count"]

    # WDMA: Weighted Deviation from Mean Agreement
    # Higher weight on deviations for sparse items (squared inverse frequency)
    merged["wdma_contrib"] = np.abs(merged["rating"] - merged["item_mean"]) / (merged["item_count"] ** 2)

    lit_feats = merged.groupby("user").agg(
        rdma=("rdma_contrib", "mean"),
        wdma=("wdma_contrib", "mean"),
    )

    # LengthVar: how much profile length deviates from average
    user_lens = df.groupby("user").size()
    lit_feats["length_var"] = ((user_lens - avg_profile_len) ** 2) / avg_profile_len
    lit_feats["length_var"] = lit_feats["length_var"].fillna(0)

    # DegSim: Average Pearson correlation with top-k nearest neighbors
    # This is expensive for large datasets, so we use a cheaper approximation:
    # Cosine similarity in SVD space with top-20 neighbors
    from sklearn.metrics.pairwise import cosine_similarity

    # Build SVD for ALL users combined (for cross-set similarity)
    all_users_in_df = sorted(df["user"].unique())
    all_users_all = sorted(all_df["user"].unique())
    uid_all = {u: i for i, u in enumerate(all_users_all)}

    rows_all = all_df["user"].map(uid_all).values
    cols_all = all_df["item"].values
    vals_all = all_df["rating"].values.astype(float)
    um_all = csr_matrix((vals_all, (rows_all, cols_all)), shape=(len(all_users_all), n_items))

    # Compute per-user similarity to top-20 neighbors using the full user-item matrix
    # For the users in df, find their rows in the full matrix
    user_indices = [uid_all[u] for u in all_users_in_df if u in uid_all]
    if len(user_indices) > 0:
        um_subset = um_all[user_indices]
        sims_matrix = cosine_similarity(um_subset, um_all)
        # For each user, get mean similarity of top-20 (excluding self)
        degsim_vals = []
        for i in range(sims_matrix.shape[0]):
            row = sims_matrix[i].copy()
            row[user_indices[i]] = -1  # exclude self
            top20 = np.sort(row)[-20:]
            degsim_vals.append(np.mean(top20))
        lit_feats["degsim"] = pd.Series(degsim_vals, index=all_users_in_df)
    else:
        lit_feats["degsim"] = 0

    # MeanVar: Variance of per-user mean rating across items
    # (attackers tend to have unusual mean ratings)
    all_user_means = all_df.groupby("user")["rating"].mean()
    global_mean_of_means = all_user_means.mean()
    global_std_of_means = all_user_means.std()
    user_means = df.groupby("user")["rating"].mean()
    lit_feats["mean_var"] = ((user_means - global_mean_of_means) / global_std_of_means) ** 2

    lit_feats = lit_feats.fillna(0)
    feats.append(lit_feats)

    # Combine all
    result = feats[0]
    for f in feats[1:]: result = result.join(f, how="left")
    result = result.fillna(0).replace([np.inf, -np.inf], 0)
    return result, svd_model

print("Computing features...")
X_train_f, svd_m = compute_features(train_ratings, all_ratings, fit_svd=True)
X_test_f, _ = compute_features(test_ratings, all_ratings, svd_model=svd_m, fit_svd=False)
y_train = labels.loc[X_train_f.index, "label"]
print(f"Base features: {X_train_f.shape}")


In [ ]:
# ============================================================
# 3. UNSUPERVISED SCORES (standard + novelty on normal-only)
# ============================================================
sc_u = RobustScaler()
Xts = sc_u.fit_transform(X_train_f)
Xes = sc_u.transform(X_test_f)

# Standard (same as v2)
iso = IsolationForest(n_estimators=200, contamination=0.09, random_state=42, max_features=0.8)
iso.fit(Xts)
X_train_f["iso_score"] = -iso.score_samples(Xts)
X_test_f["iso_score"] = -iso.score_samples(Xes)

lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.09)
lof.fit(Xts)
X_train_f["lof_score"] = -lof.score_samples(Xts)
X_test_f["lof_score"] = -lof.score_samples(Xes)

# Novelty: trained on normal only
X_normal = X_train_f.loc[normal_users, X_train_f.columns.difference(["iso_score","lof_score"])]
sc_n = RobustScaler()
Xn_s = sc_n.fit_transform(X_normal)

# Need to transform all train/test on same cols
ncols = X_train_f.columns.difference(["iso_score","lof_score"]).tolist()
Xa_s = sc_n.transform(X_train_f[ncols])
Xt_s = sc_n.transform(X_test_f[ncols])

iso_nov = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, max_features=0.7)
iso_nov.fit(Xn_s)
X_train_f["iso_novelty"] = -iso_nov.score_samples(Xa_s)
X_test_f["iso_novelty"] = -iso_nov.score_samples(Xt_s)

lof_nov = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.05)
lof_nov.fit(Xn_s)
X_train_f["lof_novelty"] = -lof_nov.score_samples(Xa_s)
X_test_f["lof_novelty"] = -lof_nov.score_samples(Xt_s)

print(f"Final features: {X_train_f.shape}")


In [ ]:
# ============================================================
# 4. MODELS (v2 models but with stronger regularization)
# ============================================================
fcols = X_train_f.columns.tolist()
X = X_train_f[fcols].values
Xt = X_test_f[fcols].values
y = y_train.values

sc2 = RobustScaler()
Xs = sc2.fit_transform(X)
Xts2 = sc2.transform(Xt)

spw = (len(y) - y.sum()) / y.sum()

def get_models():
    return {
        "lgbm_a": lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=4, num_leaves=15,
            min_child_samples=12, subsample=0.8, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=spw,
            random_state=42, verbose=-1),
        "lgbm_b": lgb.LGBMClassifier(
            n_estimators=250, learning_rate=0.04, max_depth=3, num_leaves=10,
            min_child_samples=18, subsample=0.7, colsample_bytree=0.5,
            reg_alpha=2.0, reg_lambda=3.0, scale_pos_weight=spw,
            random_state=77, verbose=-1),
        "lgbm_c": lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.035, max_depth=5, num_leaves=18,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=123, verbose=-1),
        "xgb_a": xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=4,
            min_child_weight=6, subsample=0.8, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=spw,
            random_state=42, eval_metric="logloss", verbosity=0),
        "xgb_b": xgb.XGBClassifier(
            n_estimators=250, learning_rate=0.04, max_depth=3,
            min_child_weight=10, subsample=0.7, colsample_bytree=0.5,
            reg_alpha=2.0, reg_lambda=3.0, scale_pos_weight=spw,
            random_state=77, eval_metric="logloss", verbosity=0),
        "rf": RandomForestClassifier(
            n_estimators=300, max_depth=7, min_samples_leaf=5,
            class_weight="balanced", random_state=42, n_jobs=-1),
        "gb": GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.04, max_depth=3,
            min_samples_leaf=10, subsample=0.8, random_state=42),
        "lr": LogisticRegression(
            C=0.3, class_weight="balanced", max_iter=3000, random_state=42),
    }


In [ ]:
# ============================================================
# 5. CROSS-VALIDATION (7-fold, same as v2)
# ============================================================
N_FOLDS = 7
model_names = list(get_models().keys())
oof = {n: np.zeros(len(y)) for n in model_names}
test_p = {n: np.zeros(len(Xt)) for n in model_names}

print(f"\nRunning {N_FOLDS}-fold CV...")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for fi, (tidx, vidx) in enumerate(skf.split(X, y)):
    for name, model in get_models().items():
        if name == "lr":
            model.fit(Xs[tidx], y[tidx])
            oof[name][vidx] = model.predict_proba(Xs[vidx])[:, 1]
            test_p[name] += model.predict_proba(Xts2)[:, 1] / N_FOLDS
        else:
            model.fit(X[tidx], y[tidx])
            oof[name][vidx] = model.predict_proba(X[vidx])[:, 1]
            test_p[name] += model.predict_proba(Xt)[:, 1] / N_FOLDS

print("\nOOF Results:")
for n in model_names:
    auc = roc_auc_score(y, oof[n])
    bf = max(f1_score(y, (oof[n] >= t).astype(int)) for t in np.arange(0.01, 0.99, 0.005))
    print(f"  {n:8s}: AUC={auc:.4f}, Best F1={bf:.4f}")


In [ ]:
# ============================================================
# 6. ENSEMBLE SELECTION (NO STACKING — simple averages only)
# ============================================================
print("\nEnsemble selection (NO stacking)...")

def best_f1_thr(scores, y_true):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y_true, (scores >= t).astype(int))
        if f > bf: bf, bt = f, t
    return bf, bt

results = {}
aucs = {n: roc_auc_score(y, oof[n]) for n in model_names}
sorted_m = sorted(aucs, key=aucs.get, reverse=True)

# Individual
for n in model_names:
    f1, thr = best_f1_thr(oof[n], y)
    results[f"solo_{n}"] = (f1, thr, {n: 1.0})

# Top-N simple averages and AUC-weighted
for top_n in [3, 4, 5, 6]:
    top = sorted_m[:top_n]
    oof_e = np.mean([oof[n] for n in top], axis=0)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_eq"] = (f1, thr, {n: 1/top_n for n in top})

    ws = {n: aucs[n]**3 for n in top}; sw = sum(ws.values())
    ws = {n: w/sw for n, w in ws.items()}
    oof_e = sum(ws[n] * oof[n] for n in top)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_auc3"] = (f1, thr, ws)

    oof_ranks = {n: rankdata(oof[n]) / len(y) for n in top}
    oof_e = sum(ws[n] * oof_ranks[n] for n in top)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_rank"] = (f1, thr, {**ws, "__rank__": True})

print("\nTop 15 configurations:")
for i, (name, (f1, thr, ws)) in enumerate(sorted(results.items(), key=lambda x: -x[1][0])[:15]):
    marker = " <<<" if i == 0 else ""
    print(f"  {i+1:2d}. {name:20s}: F1={f1:.4f} @ thr={thr:.3f}{marker}")

best_name = max(results, key=lambda x: results[x][0])
best_f1, best_thr, best_ws = results[best_name]
print(f"\n>>> Selected: {best_name} (F1={best_f1:.4f}, threshold={best_thr:.3f})")


In [ ]:
# ============================================================
# 7. GENERATE PREDICTIONS
# ============================================================
if "__rank__" in best_ws:
    ws = {k: v for k, v in best_ws.items() if not k.startswith("__")}
    final_oof = sum(ws[n] * rankdata(oof[n]) / len(y) for n in ws)
    final_test = sum(ws[n] * rankdata(test_p[n]) / len(Xt) for n in ws)
else:
    ws = {k: v for k, v in best_ws.items() if not k.startswith("__")}
    final_oof = sum(ws[n] * oof[n] for n in ws)
    final_test = sum(ws[n] * test_p[n] for n in ws)

# Use MEDIAN of per-fold thresholds (more robust than global optimal)
thr_folds = []
skf3 = StratifiedKFold(n_splits=10, shuffle=True, random_state=77)
for ti, vi in skf3.split(np.zeros(len(y)), y):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y[vi], (final_oof[vi] >= t).astype(int))
        if f > bf: bf, bt = f, t
    thr_folds.append(bt)

median_thr = np.median(thr_folds)
print(f"\nThreshold: global={best_thr:.3f}, median_fold={median_thr:.3f}, mean_fold={np.mean(thr_folds):.3f}")
final_thr = median_thr  # USE MEDIAN (more robust)

preds = (final_oof >= final_thr).astype(int)
print(f"\nOOF: AUC={roc_auc_score(y, final_oof):.4f}, F1={f1_score(y, preds):.4f}, "
      f"P={precision_score(y, preds):.4f}, R={recall_score(y, preds):.4f}")
print(classification_report(y, preds, target_names=["Normal", "Anomalous"]))


In [ ]:
# ============================================================
# 8. SAVE
# ============================================================
test_users = X_test_f.index.values
pred_dict = {int(u): float(s) for u, s in zip(test_users, final_test)}

np.savez("submission.npz", predictions=final_test)
with open("predictions.json", "w") as f:
    json.dump({"predictions": pred_dict}, f)
pd.DataFrame({"user": test_users, "anomaly_score": final_test,
    "predicted_label": (final_test >= final_thr).astype(int)
}).sort_values("anomaly_score", ascending=False).to_csv("predictions.csv", index=False)

n_anom = (final_test >= final_thr).sum()
print(f"Test: {len(test_users)} users, {n_anom} predicted anomalies ({n_anom/len(test_users)*100:.1f}%)")
print(f"Score: mean={final_test.mean():.4f}, >0.5: {(final_test>=0.5).sum()}, >0.3: {(final_test>=0.3).sum()}")
print("\n✓ DONE")